# Trader analyst assignment

A trader’s day-to-day work involves extracting insights from data, suggesting improvements to existing trading strategies, and developing new ones.
This assignment covers both: Part 1 focuses on finding new strategies, and Part 2 focuses on improving live strategies. Each part includes its own dataset and questions to answer. You’re not limited to answering only our questions, there’s always more to discover, so feel free to share any additional insights you find.

In this assignment, **reference price** and **fair value** are interchangeable terms.

Good luck, and keep in mind that your approach and thought process matter more than the final numeric results.

# Part 1: Gaining trading insights from trade data

You are given executed trade data and reference price data for two markets: BTC/USDT and ETH/USDT.
The datasets cover the same time period and allow you to evaluate trading performance at trade level. Your goal is to compute Profit and Loss (PnL) over different horizons, diagnose why one market is clearly underperforming, and propose concrete improvements to the trading strategy.

## Profit and Loss
A core metric tracked by traders is **Profit and Loss (PnL)**. In this assignment, you will compute PnL using two complementary approaches:
- **Theoretical PnL**: Based on executed trades and reference prices over a chosen time horizon
  (e.g. at trade time or 1 second later). This measures execution quality and short-term slippage. We calculate this PnL as follows:
$$
\mathrm{PnL}_t(\text{side}) =
\text{trade amount} \times s \times (\text{reference price}_t - \text{trade price})
- \text{fee amount} \times \text{trade price},
\quad
s =
\begin{cases}
+1 & \text{if side = buy} \\
-1 & \text{if side = sell}
\end{cases}
$$
- **Accounting PnL**: Based on realized cash flows from trades and the value of the final inventory at the end of the dataset. This represents the actual profit or loss of the strategy over the full trading period.

Both perspectives are important and may lead to different conclusions.

## Questions

Address the following questions in your analysis:

- Merge the trade data with the price data and compute the theoretical PnL for different time horizons (e.g. for 1, 5 and 30 seconds). What patterns do you observe?
- One of the two markets is clearly underperforming. Can you identify the main source of the PnL leak? What mistake is being made by us?
- Break down performance across different dimensions (e.g. side, trigger, indicator, maker/taker). What concrete changes would you recommend to improve the strategy?
- What do you think the `indicator` column represents?
- What is the accounting PnL over the full dataset time span? How does it compare to the theoretical PnL results?

## Data

Files are in `data/part_1`:

- `trade_btcusdt_*.pickle` and `trade_ethusdt_*.pickle` — hourly executed-trade
  data for BTC/USDT and ETH/USDT, including timestamps, prices, sizes, fees,
  sides, and trade metadata.
- `price_btcusdt_*.pickle` and `price_ethusdt_*.pickle` — hourly reference-price
  snapshots for the same two markets.

Load the relevant hourly files for each market. All timestamps are UTC and all
prices are denominated in USDT.


## Answers — Part 1

### Question 1: Merge the data and compute theoretical PnL

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

P1 = Path("data/part_1")

def market_from_file(path):
    symbol = Path(path).name.split("_")[1].upper()
    return f"{symbol[:3]}-{symbol[3:]}"

def load_pickles(pattern):
    frames = []
    for path in sorted(P1.glob(pattern)):
        frame = pd.read_pickle(path).copy()
        frame["time"] = pd.to_datetime(frame["time"], utc=True)
        frame["market"] = market_from_file(path)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(pattern)
    return pd.concat(frames, ignore_index=True)

trades = load_pickles("trade_*usdt_*.pickle").sort_values(["market", "time"])
prices = (load_pickles("price_*usdt_*.pickle")
          .rename(columns={"mid_price": "reference_price"})
          .sort_values(["market", "time"]))

def theoretical_pnl_at(horizon_seconds):
    pieces = []
    for market, t in trades.groupby("market", sort=False):
        p = prices.loc[prices.market.eq(market), ["time", "reference_price"]]
        x = t.copy()
        x["target_time"] = x["time"] + pd.Timedelta(seconds=horizon_seconds)
        x = pd.merge_asof(
            x.sort_values("target_time"), p.sort_values("time"),
            left_on="target_time", right_on="time", direction="nearest",
            tolerance=pd.Timedelta(seconds=1), suffixes=("", "_reference")
        )
        x["horizon_seconds"] = horizon_seconds
        x["side_sign"] = x["own_side"].map({"buy": 1.0, "sell": -1.0})
        x["theoretical_pnl"] = (
            x["trade_amount"] * x["side_sign"]
            * (x["reference_price"] - x["trade_price"])
            - x["fee_amount"] * x["trade_price"]
        )
        pieces.append(x)
    return pd.concat(pieces, ignore_index=True)

horizons = [0, 1, 5, 30]
pnl = pd.concat([theoretical_pnl_at(h) for h in horizons], ignore_index=True)
coverage = pnl.groupby(["horizon_seconds", "market"])["reference_price"].apply(lambda s: s.notna().mean())
summary = pnl.pivot_table(index="horizon_seconds", columns="market", values="theoretical_pnl", aggfunc="sum")
summary["Overall"] = summary.sum(axis=1)
display(summary.round(2))
print("Reference-price match coverage:")
display(coverage.unstack().round(4))

market,BTC-USDT,ETH-USDT,Overall
horizon_seconds,,,
0,316.07,-357.82,-41.74
1,276.09,-391.57,-115.48
5,309.27,-376.02,-66.75
30,301.06,-397.59,-96.54


Reference-price match coverage:


market,BTC-USDT,ETH-USDT
horizon_seconds,,
0,0.9925,0.9945
1,0.9935,0.9945
5,0.9944,0.9945
30,0.9878,0.9959


### Results Interpretation

The theoretical-PnL results show a clear and persistent separation between the two markets:

| Horizon | BTC/USDT | ETH/USDT | Combined |
|---:|---:|---:|---:|
| 0 seconds | +316.07 | -357.82 | -41.74 |
| 1 second | +276.09 | -391.57 | -115.48 |
| 5 seconds | +309.27 | -376.02 | -66.75 |
| 30 seconds | +301.06 | -397.59 | -96.54 |

BTC/USDT is profitable at every horizon and its result is reasonably stable. This indicates that BTC executions generally occur at favorable prices relative to fair value and retain most of that edge after execution.

ETH/USDT is negative at every horizon. Importantly, most of the ETH loss is already present at 0 seconds. That is primarily an execution-time or quote-calibration problem: the strategy is trading at prices that are unfavorable relative to contemporaneous fair value. ETH then deteriorates by approximately 33.75 USDT between 0 and 1 second and by approximately 39.77 USDT between 0 and 30 seconds. This later deterioration is the cleaner evidence of post-trade adverse selection.

Therefore, ETH has two related problems:

1. a large initial execution/fair-value deficit; and
2. additional unfavorable price movement after execution.

Reference-price coverage is approximately 99% under the one-second tolerance. The unmatched observations are excluded from PnL sums rather than being joined to potentially stale prices.

### Question 2: Identify the source of the PnL leak

### Questions 3–4: Performance breakdown, recommendations, and interpretation of `indicator`

In [3]:
trade_features = trades.copy()
trade_features["indicator_numeric"] = pd.to_numeric(trade_features["indicator"], errors="coerce")
trade_features.loc[trade_features["indicator_numeric"].eq(-9999), "indicator_numeric"] = np.nan
trade_features["indicator_band"] = "invalid/missing"
for market, idx in trade_features.groupby("market").groups.items():
    valid_idx = trade_features.loc[idx, "indicator_numeric"].dropna().index
    trade_features.loc[valid_idx, "indicator_band"] = pd.qcut(
        trade_features.loc[valid_idx, "indicator_numeric"], 5,
        labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"], duplicates="drop"
    ).astype(str)

feature_cols = ["market", "time", "trade_price", "trade_amount", "own_side", "trigger",
                "taker_side", "counterparty_id", "indicator_band"]
pnl = pnl.drop(columns=["indicator_band"], errors="ignore").merge(
    trade_features[feature_cols],
    on=["market", "time", "trade_price", "trade_amount", "own_side", "trigger",
        "taker_side", "counterparty_id"], how="left", validate="many_to_one",
    suffixes=("", "_feature")
)

def breakdown(frame, dimensions):
    return (frame.groupby(dimensions, dropna=False)["theoretical_pnl"]
            .agg(total_pnl="sum", total_rows="size", matched_trades="count", pnl_per_matched_trade="mean")
            .sort_values("total_pnl"))

five = pnl.loc[pnl.horizon_seconds.eq(5)]
print("5-second PnL by market and side")
display(breakdown(five, ["market", "own_side"]).round(3))
print("5-second PnL by market and trigger")
display(breakdown(five, ["market", "trigger"]).round(3))
print("5-second PnL by market and taker direction")
display(breakdown(five, ["market", "taker_side"]).round(3))
print("5-second PnL by market and indicator band")
display(breakdown(five, ["market", "indicator_band"]).round(3))
print("Worst ETH counterparties at 5 seconds")
display(breakdown(five.loc[five.market.eq("ETH-USDT")], ["counterparty_id"]).head(10).round(3))
print("ETH trigger × own-side PnL at 5 seconds")
display(five.loc[five.market.eq("ETH-USDT")].pivot_table(
    index="trigger", columns="own_side", values="theoretical_pnl", aggfunc="sum").round(3))

5-second PnL by market and side


total_pnl  total_rows  matched_trades  \
market   own_side                                          
ETH-USDT buy        -331.803         316             315   
         sell        -44.213         305             303   
BTC-USDT NaN           0.000          84               0   
ETH-USDT NaN           0.000         102               0   
BTC-USDT buy          32.828         283             281   
         sell        276.437         702             698   

                   pnl_per_matched_trade  
market   own_side                         
ETH-USDT buy                      -1.053  
         sell                     -0.146  
BTC-USDT NaN                         NaN  
ETH-USDT NaN                         NaN  
BTC-USDT buy                       0.117  
         sell                      0.396

5-second PnL by market and trigger


,,total_pnl,total_rows,matched_trades,pnl_per_matched_trade
market,trigger,,,,
ETH-USDT,3.0,-372.231,216,216,-1.723
BTC-USDT,3.0,-113.307,325,323,-0.351
ETH-USDT,1.0,-51.721,196,195,-0.265
BTC-USDT,NaN,0.000,84,0,NaN
ETH-USDT,NaN,0.000,102,0,NaN
BTC-USDT,1.0,24.649,324,320,0.077
ETH-USDT,2.0,47.935,209,207,0.232
BTC-USDT,2.0,397.924,336,336,1.184


5-second PnL by market and taker direction


total_pnl  total_rows  matched_trades  \
market   taker_side                                          
ETH-USDT sell         -268.162         366             311   
         buy          -107.855         357             307   
BTC-USDT sell           29.454         277             229   
         buy           279.812         792             750   

                     pnl_per_matched_trade  
market   taker_side                         
ETH-USDT sell                       -0.862  
         buy                        -0.351  
BTC-USDT sell                        0.129  
         buy                         0.373

... (2 outputs left out)

Worst ETH counterparties at 5 seconds


,total_pnl,total_rows,matched_trades,pnl_per_matched_trade
counterparty_id,,,,
CP-007,-65.015,33,26,-2.501
CP-011,-64.842,130,113,-0.574
CP-013,-64.645,112,95,-0.680
CP-004,-40.436,166,143,-0.283
CP-008,-35.152,26,20,-1.758
CP-009,-27.432,80,71,-0.386
CP-005,-25.327,12,10,-2.533
CP-012,-20.369,21,16,-1.273
CP-003,-15.752,66,56,-0.281


ETH trigger × own-side PnL at 5 seconds


own_side,buy,sell
trigger,,
1.0,-39.827,-11.894
2.0,4.819,43.116
3.0,-296.796,-75.435


In [4]:
marks = pnl.pivot_table(
    index=["market", "time", "trade_price", "trade_amount", "own_side", "trigger"],
    columns="horizon_seconds", values="theoretical_pnl", aggfunc="first"
).reset_index()
for h in [1, 5, 30]:
    marks[f"post_trade_change_{h}s"] = marks[h] - marks[0]

diagnostic = marks.groupby(["market", "trigger"]).agg(
    pnl_0s=(0, "sum"), pnl_5s=(5, "sum"), pnl_30s=(30, "sum"),
    post_trade_change_5s=("post_trade_change_5s", "sum"),
    post_trade_change_30s=("post_trade_change_30s", "sum"), trades=(0, "size")
)
display(diagnostic.round(3))

# Mechanical sign-flip sensitivity only; this is not an executable backtest.
eth_t3 = five.loc[five.market.eq("ETH-USDT") & five.trigger.eq(3)]
observed = eth_t3.theoretical_pnl.sum()
fee_cost = (eth_t3.fee_amount * eth_t3.trade_price).sum()
gross_before_fee = observed + fee_cost
mechanical_reversal = -gross_before_fee - fee_cost
print(f"Observed ETH trigger-3 PnL at 5s: {observed:,.2f} USDT")
print(f"Mechanical sign-flip sensitivity: {mechanical_reversal:,.2f} USDT")

pnl_0s   pnl_5s  pnl_30s  post_trade_change_5s  \
market   trigger                                                    
BTC-USDT 1.0       32.159   24.649   34.120                -7.464   
         2.0      361.070  397.924  384.540                36.574   
         3.0      -77.157 -113.307 -117.604               -36.087   
ETH-USDT 1.0      -31.299  -51.721  -59.448               -19.937   
         2.0       23.765   47.935   49.704                24.112   
         3.0     -350.280 -372.231 -387.848               -21.951   

                  post_trade_change_30s  trades  
market   trigger                                 
BTC-USDT 1.0                      2.125     324  
         2.0                     27.295     336  
         3.0                    -40.237     325  
ETH-USDT 1.0                    -27.380     196  
         2.0                     25.939     209  
         3.0                    -38.331     216

Observed ETH trigger-3 PnL at 5s: -372.23 USDT
Mechanical sign-flip sensitivity: -454.21 USDT


### Results Interpretation — Question 2

#### Overall diagnosis

The underperforming market is **ETH/USDT**. Its theoretical PnL is negative at every measured horizon, while BTC/USDT remains profitable. The leak is not evenly distributed across ETH trades: it is concentrated in **trigger 3**, particularly on the **buy side**, and becomes worse in the higher `indicator` bands.

At five seconds, ETH loses **-376.02 USDT** in total. Trigger 3 alone contributes **-372.23 USDT**, or approximately **99% of the market's net loss**. Trigger 2 is profitable and offsets part of the losses from triggers 1 and 3.

#### ETH PnL by trigger and horizon

| ETH trigger | 0-second PnL | 5-second PnL | 30-second PnL | 0s → 30s change | Interpretation |
|---:|---:|---:|---:|---:|---|
| 1 | -31.30 | -51.72 | -58.68 | -27.38 | Moderately negative and deteriorates after execution |
| 2 | +23.77 | +47.94 | +49.70 | +25.94 | Profitable and improves after execution |
| 3 | -350.28 | -372.23 | -388.61 | -38.33 | Dominant loss source; poor initially and afterward |
| **Total** | **-357.82** | **-376.02** | **-397.59** | **-39.77** | **Large execution deficit plus adverse markout** |

Trigger 3 is already deeply negative at 0 seconds. Therefore, most of its loss is not caused by a price move occurring several seconds after the trade; the strategy is already trading at an unfavorable level relative to contemporaneous fair value. The further deterioration to 30 seconds shows that the strategy also experiences adverse post-trade movement.

Trigger 2 behaves in the opposite way. It starts positive and becomes more profitable at longer horizons, suggesting favorable execution and favorable subsequent markout. This is why disabling the whole ETH strategy would be too broad: it would remove the only consistently profitable trigger in this sample.

#### Interaction between trigger and own side

| ETH trigger at 5s | Buy PnL | Sell PnL | Total PnL | Main observation |
|---:|---:|---:|---:|---|
| 1 | -39.83 | -11.89 | -51.72 | Both sides lose; buys are worse |
| 2 | +4.82 | +43.12 | +47.94 | Both sides make money; sells dominate |
| 3 | -296.80 | -75.44 | -372.23 | Both sides lose; buys are the main leak |
| **Total** | **-331.80** | **-44.21** | **-376.02** | **Approximately 88% of net ETH loss comes from buys** |

The interaction is more informative than looking at either trigger or side alone. ETH buys are not universally bad: trigger-2 buys are profitable. Likewise, trigger 3 is not only a buy-side issue because its sells also lose. The worst regime is specifically the combination of **ETH + trigger 3 + own-side buy**.

#### Execution deficit versus adverse selection

The two effects should be separated:

| Effect | Evidence | Interpretation |
|---|---|---|
| Execution-time deficit | ETH trigger 3 is -350.28 USDT at 0s | Trades are unfavorable relative to contemporaneous fair value immediately |
| Post-trade adverse selection | Trigger 3 deteriorates by another 38.33 USDT by 30s | Fair value moves further against the position after execution |
| Favorable post-trade movement | Trigger 2 improves by 25.94 USDT by 30s | This trigger appears directionally better calibrated |

Calling the entire ETH loss “adverse selection” would therefore be imprecise. The larger component is the execution-time deficit; adverse selection adds to it afterward.

#### What mistake is likely being made?

The evidence supports a **market-specific calibration mistake**. The strategy continues to trade ETH trigger 3 even though its execution edge is substantially negative, its post-trade markout is unfavorable, and its buy-side losses become severe in higher indicator regimes. Possible implementation causes include:

- using a BTC-calibrated trigger or threshold for ETH;
- applying an indicator threshold with the wrong ETH scaling;
- quoting too aggressively when trigger 3 fires;
- failing to include expected fees and adverse markout in the decision threshold;
- using a stale or insufficiently responsive ETH fair-value/signal input;
- mapping a valid signal to unsuitable size or direction under high-indicator conditions.

The dataset does not reveal the strategy source code, so it cannot uniquely distinguish among these causes. The most defensible statement is: **the mistake is allowing an uneconomic ETH trigger-3 regime to trade without adequate market-, side-, and indicator-specific filtering.**

#### Does the trade direction simply need to be reversed?

| Trigger-3 sensitivity at 5s | PnL |
|---|---:|
| Observed ETH trigger-3 PnL | -372.23 USDT |
| Mechanical sign reversal after observed fees | -454.21 USDT |

A direction error is initially plausible because the regime loses consistently, but the mechanical reversal remains negative after fees. Reversing the gross directional exposure also requires paying the same observed fees, and the resulting edge is insufficient to cover them.

This is not an executable backtest: reversed orders would face different quotes, fills, market impact, and possibly fees. Nevertheless, it shows that **“flip every trigger-3 trade” is not supported as a simple fix**. The regime should first be paused or filtered, then redesigned and tested using realistic executable prices.

### Results Interpretation — Question 3

#### Performance by market and trigger

| Market and trigger | 0-second PnL | 5-second PnL | 30-second PnL | Assessment |
|---|---:|---:|---:|---|
| BTC — Trigger 1 | +32.16 | +24.65 | +34.28 | Positive but relatively small |
| BTC — Trigger 2 | +361.07 | +397.92 | +388.37 | Strongest BTC regime |
| BTC — Trigger 3 | -77.16 | -113.31 | -117.39 | Negative and deteriorating |
| ETH — Trigger 1 | -31.30 | -51.72 | -58.68 | Negative, secondary leak |
| ETH — Trigger 2 | +23.77 | +47.94 | +49.70 | Only profitable ETH trigger |
| ETH — Trigger 3 | -350.28 | -372.23 | -388.61 | Primary strategy failure |

The trigger pattern is directionally similar across markets: trigger 2 is the strongest regime and trigger 3 is the weakest. However, BTC trigger 2 is sufficiently profitable to keep BTC positive overall, while ETH trigger 3 overwhelms ETH trigger 2. This raises the possibility that trigger 3 is structurally weak but especially poorly calibrated for ETH.

BTC trigger 3 should not be ignored merely because BTC is profitable in aggregate. It loses approximately 113.31 USDT at five seconds and deteriorates further by 30 seconds. A trigger-3 audit may therefore improve both markets, with ETH as the urgent priority.

#### Performance by own side

| Market | Buy PnL at 5s | Sell PnL at 5s | Total | PnL concentration |
|---|---:|---:|---:|---|
| BTC/USDT | +32.83 | +276.44 | +309.27 | Approximately 89% comes from sells |
| ETH/USDT | -331.80 | -44.21 | -376.02 | Approximately 88% of net loss comes from buys |

BTC is profitable on both sides, which provides some evidence that its overall result is not a one-sided accident. Its sell side is nevertheless much stronger. ETH is negative on both sides, but its buy-side result is dramatically worse.

Raw side totals can be influenced by trade counts and trade sizes. The code also reports PnL per trade, but a production analysis should additionally normalize by traded notional, risk, and inventory impact before changing side-specific allocations.

#### Performance by taker direction

| Market | Taker buys: 5s PnL | Taker sells: 5s PnL | Interpretation |
|---|---:|---:|---|
| BTC/USDT | +279.81 | +29.45 | Profitable against both flow directions |
| ETH/USDT | -107.85 | -268.16 | Negative against both; worse when taker sells |

For ETH, the larger loss when the taker sells is consistent with the poor own-buy result because the liquidity provider is normally on the opposite side of the aggressor. However, `taker_side` records the aggressor's direction; it is not a direct maker/taker flag for our own order. The analysis should therefore be described as **performance by taker direction**, not as a definitive maker-versus-taker classification.

#### Indicator-band performance

Indicator bands are calculated once on unique trades and separately within each market. Invalid observations include missing values, the `-9999` sentinel, and the text calculation error.

| Indicator band | BTC 5s PnL | ETH 5s PnL | ETH total rows | ETH price-matched trades | ETH PnL per matched trade | Interpretation |
|---|---:|---:|---:|---:|---:|---|
| Q1 lowest | +11.46 | -21.09 | 114 | 114 | -0.185 | ETH loss is relatively mild |
| Q2 | +13.15 | -29.51 | 114 | 113 | -0.261 | ETH worsens |
| Q3 | +44.12 | -40.10 | 113 | 113 | -0.355 | Continued deterioration |
| Q4 | +169.77 | -77.26 | 114 | 114 | -0.678 | BTC strongest; ETH materially weak |
| Q5 highest | +50.79 | -163.20 | 114 | 113 | -1.444 | Worst ETH valid-signal regime |
| Invalid/missing | +19.98 | -44.87 | 154 | 51 | -0.880 | Missing/error states also carry risk |

ETH PnL becomes progressively worse as the indicator rises. The highest quintile loses almost eight times as much per trade as the lowest quintile. BTC does not show the same monotonic deterioration and is strongest in Q4. This is important evidence that the signal's calibration or economic meaning differs by market.

The invalid/missing category also performs poorly for ETH. A failed signal calculation should not silently fall back to normal trading behavior. It should trigger an explicit safe policy such as no trade, reduced size, or a separate fallback model.

#### Counterparty concentration

| Rank | ETH counterparty | 5-second PnL | Total rows | Price-matched trades | PnL per matched trade | Interpretation |
|---:|---|---:|---:|---:|---:|---|
| 1 | CP-007 | -65.02 | 33 | 26 | -2.50 | Small matched sample, very poor unit economics |
| 2 | CP-011 | -64.84 | 130 | 113 | -0.57 | Persistent high-count loss |
| 3 | CP-013 | -64.65 | 112 | 95 | -0.68 | Persistent high-count loss |
| 4 | CP-004 | -40.44 | 166 | 143 | -0.28 | Largest matched count among major losers |
| 5 | CP-008 | -35.15 | 26 | 20 | -1.76 | Concentrated poor outcomes |

CP-007 is especially notable because it produces a large loss from relatively few trades. CP-011 and CP-013 lose similar totals across many more observations, suggesting a more persistent but less extreme per-trade pattern.

Counterparty PnL is diagnostic, not causal. These counterparties may simply select the strategy's weakest quotes or trade during the bad trigger-3/high-indicator regime. Before imposing restrictions, counterparty performance should be conditioned on trigger, side, indicator, size, and time. A global counterparty block risks removing profitable interactions outside the problematic regime.

#### Concrete strategy improvements

| Priority | Proposed change | Evidence | Expected benefit | Main validation risk |
|---:|---|---|---|---|
| 1 | Pause or sharply reduce ETH trigger 3 | -372.23 USDT at 5s | Stops the dominant observed leak | Sample may not represent other periods |
| 2 | Preserve and validate ETH trigger 2 | +47.94 USDT at 5s | Retains the profitable ETH regime | Possible overfitting to one day |
| 3 | Add an ETH high-indicator filter | Q5 loses -163.20 USDT | Avoids the worst valid-signal state | Threshold instability |
| 4 | Use a safe fallback for invalid indicators | Invalid/missing loses -44.87 USDT | Prevents trading without a valid signal | Reduced participation |
| 5 | Recalibrate ETH buy thresholds and size | Buys lose -331.80 USDT | Targets the largest side-level leak | Side result may reflect regime mix |
| 6 | Audit trigger 3 on BTC as well | BTC trigger 3 loses -113.31 USDT | Potential improvement beyond ETH | Could reduce profitable offsets elsewhere |
| 7 | Add counterparty-regime monitoring | Losses concentrated in several IDs | Detects informed/adverse flow | Counterparty sample sizes vary |
| 8 | Include fees in entry thresholds | Reversal remains negative after fees | Prevents gross signals with negative net value | Fee schedule may change |

Any rule change should be tested on additional dates and evaluated using an executable simulation. The evaluation should include spread, fees, partial fills, order size, queue position, market impact, inventory constraints, and separate 0-, 1-, 5-, and 30-second markouts.

### Results Interpretation — Question 4

#### What does the `indicator` likely represent?

The `indicator` is most likely a **continuous pre-trade market-state or model signal**, rather than a price, fee, or post-trade accounting field. Evidence for this interpretation includes:

| Observation | What it suggests |
|---|---|
| Most valid values are small continuous numbers | A normalized feature, score, return, imbalance, or threshold variable |
| Values differ materially between BTC and ETH | Market-specific scale or calibration |
| PnL changes systematically across indicator bands | The feature is related to the strategy's trading regime |
| Some rows contain `-9999` | A conventional missing/error sentinel |
| Some rows contain a calculation-error string | The value is generated by a model or feature pipeline |
| Some values are missing | The strategy sometimes trades without a usable calculation or uses a fallback path |

Plausible definitions include:

- short-horizon momentum or return;
- order-book imbalance;
- deviation between venue price and reference fair value;
- normalized spread or basis;
- volatility or trade-intensity score;
- model confidence or signal strength.

The column's exact semantic meaning cannot be recovered from these trade records alone. A high value being associated with poor ETH PnL does not tell us whether the feature itself is wrong, its sign is wrong, its threshold is wrong, or another correlated market regime causes the loss.

#### Market-dependent behavior

BTC and ETH react differently to the same relative indicator bands. BTC is profitable across all bands and strongest in Q4, while ETH becomes increasingly negative toward Q5. This makes two interpretations plausible:

1. the indicator captures a useful condition, but ETH maps it to an unsuitable quote, size, or direction; or
2. the raw indicator scale/normalization is not comparable across markets and ETH thresholds were copied from another market.

Because the quintiles are constructed separately by market, the result is not merely caused by ETH having numerically larger raw values. It reflects poorer ETH performance at the high end of ETH's own distribution.

#### How to identify the indicator more confidently

The feature should be tested against candidate variables using only information available before each trade:

1. correlate it with past and future mid-price returns over multiple horizons;
2. compare it with bid/ask spread and order-book imbalance;
3. test its relationship with realized volatility and trade intensity;
4. examine its sign and distribution by trigger and own side;
5. determine whether missing/error values activate a fallback trading rule;
6. inspect the feature-generation code or data dictionary if available.

The safest written conclusion is therefore: **`indicator` appears to be a pre-trade continuous signal or market-state feature whose ETH calibration is problematic, particularly at high values and in trigger 3. Its precise definition cannot be proven from the supplied columns alone.**


### Question 5: Accounting PnL vs Theoretical PnL

In [5]:
final_reference = prices.groupby("market")["reference_price"].last()
accounting = trades.copy()
accounting["final_reference_price"] = accounting["market"].map(final_reference)
accounting["side_sign"] = accounting["own_side"].map({"buy": 1.0, "sell": -1.0})
accounting["accounting_pnl"] = (
    accounting["trade_amount"] * accounting["side_sign"]
    * (accounting["final_reference_price"] - accounting["trade_price"])
    - accounting["fee_amount"] * accounting["trade_price"]
)
accounting_summary = accounting.groupby("market")["accounting_pnl"].sum()
comparison = summary.drop(columns="Overall").T.join(accounting_summary).rename(
    columns={0: "theoretical_0s", 1: "theoretical_1s", 5: "theoretical_5s",
             30: "theoretical_30s"})
display(comparison.round(2))
print(f"Combined accounting PnL: {accounting.accounting_pnl.sum():,.2f} USDT")

,theoretical_0s,theoretical_1s,theoretical_5s,theoretical_30s,accounting_pnl
market,,,,,
BTC-USDT,316.07,276.09,309.27,301.06,51.08
ETH-USDT,-357.82,-391.57,-376.02,-397.59,1561.28


Combined accounting PnL: 1,612.36 USDT


### Results Interpretation

Assuming zero opening strategy inventory and cash, accounting PnL is equivalent to trade cash flow plus the final marked value of the net acquired inventory.

| Market | Theoretical PnL (0s) | Theoretical PnL (1s) | Theoretical PnL (5s) | Theoretical PnL (30s) | Accounting PnL |
|---|---:|---:|---:|---:|---:|
| BTC/USDT | +316.07 USDT | +276.09 USDT | +309.27 USDT | +301.06 USDT | +51.08 USDT |
| ETH/USDT | -357.82 USDT | -391.57 USDT | -376.02 USDT | -397.59 USDT | +1,561.28 USDT |
| **Combined** | **-41.74 USDT** | **-115.48 USDT** | **-66.75 USDT** | **-96.54 USDT** | **+1,612.36 USDT** |

This differs materially from the short-horizon theoretical results. BTC has strong short-horizon execution PnL but a much smaller full-period accounting gain. ETH has negative short-horizon PnL but a large positive accounting result.

The measures answer different questions:

- **Theoretical PnL** measures execution quality and short-term markout relative to nearby fair value.
- **Accounting PnL** measures the final marked value of all trading cash flows and accumulated inventory.

ETH's positive accounting result does not invalidate the diagnosed execution leak. It means that the net ETH inventory accumulated through trading benefited from the subsequent full-period price move. A strategy can therefore make poor individual executions yet finish with positive accounting PnL because it carried favorable directional exposure.

# Part 2: COIN across multiple exchanges

You traded COIN across several exchanges over one week. Prices generally track each other, but venue dislocations occur. COIN is sometimes bought on one venue and moved to another. The data contains both market activity and our own trading activity.

Your objective is to determine what happened, reconcile the portfolio, and explain
which conclusions are robust versus dependent on an accounting assumption. Show your
reasoning and material intermediate results; the quality of the analysis matters more
than one number.

## Data

Candidate data is in `data/part_2`:

- `coin_trades_all_venues.csv.gz` — public and own COIN trades. Relevant fields include
  timestamp, exchange, price, amount, `own_side`, fee, fair value, exchange rate,
  `own_trade`, and `taker_side`.
- `coin_transfers.csv` — COIN transfer requests, send and receipt timestamps, amounts,
  source/destination, amounts sent and received, and fair-value marks at send and receipt.
- `coin_opening_positions.csv` — COIN inventory recorded on each trading venue's first
  in-window trade.

All timestamps are UTC. Convert local-currency values to USDT using the supplied
exchange rate. Exchanges A–E have trading data. Other exchanges can appear in transfer
records without an observed public COIN price feed.

Some parts of the analysis require a global COIN fair-value reference. Decide how to
construct and apply it, and state your material transfer-timing and inventory assumptions.

## Questions

Answer concisely, with evidence:

1. How did you value COIN and calculate portfolio PnL? Explain and reconcile your
   consolidated and per-exchange results.
2. Was COIN trading profitable overall and on each exchange?
3. Were the transfers successful from a PnL and operational perspective?
4. Are there any errors, contradictions, or unrealistic events in the data? What is their
   impact on your analysis?
5. What were the main drivers of PnL, and which conclusions are sensitive to your
   assumptions or data-quality findings?

There is intentionally no single prescribed accounting convention. We will assess whether
your choices are explicit, economically coherent, and consistently applied.

## Answers — Part 2

### Question 1: COIN valuation and portfolio PnL

I convert venue-local prices, fair values, and monetary trading fees to USDT using:

$$
\text{USDT value} = \frac{\text{local-currency value}}{\text{exchange rate}}.
$$

Only rows with `own_trade = "t"` affect our portfolio. Buys increase COIN inventory and reduce USDT cash; sells reduce inventory and increase cash. Trading fees reduce cash on both sides.

For a common COIN mark, I use the median converted fair value across observations within each second. Opening positions are marked using the nearest global fair value within two seconds, and all final inventory is valued at the last global fair value, approximately 0.003545445 USDT per COIN.

Completed transfers change source and destination balances. A sent-but-unreceived transfer is carried as `IN_TRANSIT`. Receipt-without-send records are excluded from the base ledger because including them would create unexplained inventory.

In [5]:
P2 = Path("data/part_2")
all_trades = pd.read_csv(P2 / "coin_trades_all_venues.csv.gz")
opening = pd.read_csv(P2 / "coin_opening_positions.csv")
transfers = pd.read_csv(P2 / "coin_transfers.csv")

all_trades["time"] = pd.to_datetime(all_trades["time_utc"], utc=True)
all_trades["price_usdt"] = all_trades["price"] / all_trades["exchange_rate"]
all_trades["fair_value_usdt"] = all_trades["fair_value"] / all_trades["exchange_rate"]
all_trades["fee_usdt"] = all_trades["fee"].fillna(0) / all_trades["exchange_rate"]
own = all_trades.loc[all_trades.own_trade.eq("t")].copy()
own["signed_coin"] = np.where(own.own_side.eq("buy"), own.amount, -own.amount)
own["trade_cash_usdt"] = np.where(
    own.own_side.eq("buy"), -own.amount * own.price_usdt, own.amount * own.price_usdt
) - own.fee_usdt
own["execution_pnl_usdt"] = np.where(
    own.own_side.eq("buy"),
    own.amount * (own.fair_value_usdt - own.price_usdt),
    own.amount * (own.price_usdt - own.fair_value_usdt)
) - own.fee_usdt

snapshots = all_trades.dropna(subset=["fair_value_usdt"])[["time", "exchange", "fair_value_usdt"]].copy()
snapshots["second"] = snapshots.time.dt.floor("s")
global_fair = snapshots.groupby("second")["fair_value_usdt"].median().sort_index()
final_fair = global_fair.iloc[-1]

opening["time"] = pd.to_datetime(opening["as_of_time_utc"], utc=True)
opening = pd.merge_asof(
    opening.sort_values("time"), global_fair.rename("opening_fair_usdt").reset_index(),
    left_on="time", right_on="second", direction="nearest", tolerance=pd.Timedelta(seconds=2)
)

for c in ["time_request_utc", "time_sent_utc", "time_confirmed_received_utc"]:
    transfers[c] = pd.to_datetime(transfers[c], utc=True, errors="coerce", format="mixed")
transfers["status"] = np.select(
    [transfers.time_sent_utc.notna() & transfers.time_confirmed_received_utc.notna(),
     transfers.time_sent_utc.notna() & transfers.time_confirmed_received_utc.isna(),
     transfers.time_sent_utc.isna() & transfers.time_confirmed_received_utc.notna()],
    ["complete", "sent_not_received", "received_without_send"], default="request_only"
)
display(pd.DataFrame({
    "own_trades": [len(own)], "final_global_fair_usdt": [final_fair],
    "transfer_records": [len(transfers)]
}).round(9))

,own_trades,final_global_fair_usdt,transfer_records
0,33932,0.003545,161


In [6]:
venues = sorted(set(opening.exchange) | set(own.exchange)
                | set(transfers.exchange_from) | set(transfers.exchange_to))
ledger = pd.DataFrame(index=venues)
ledger.index.name = "location"
ledger["opening_coin"] = opening.set_index("exchange")["opening_coin_position"].reindex(venues).fillna(0)
ledger["net_trade_coin"] = own.groupby("exchange")["signed_coin"].sum().reindex(venues).fillna(0)

complete = transfers.loc[transfers.status.eq("complete")].copy()
sent_unresolved = transfers.loc[transfers.status.eq("sent_not_received")].copy()
ledger["completed_sent_coin"] = complete.groupby("exchange_from")["amount_sent"].sum().reindex(venues).fillna(0)
ledger["completed_received_coin"] = complete.groupby("exchange_to")["amount_received"].sum().reindex(venues).fillna(0)
ledger["unresolved_sent_coin"] = sent_unresolved.groupby("exchange_from")["amount_sent"].sum().reindex(venues).fillna(0)
ledger["ending_coin"] = (ledger.opening_coin + ledger.net_trade_coin
                         - ledger.completed_sent_coin + ledger.completed_received_coin
                         - ledger.unresolved_sent_coin)

in_transit_coin = sent_unresolved.amount_sent.sum()
ledger.loc["IN_TRANSIT", :] = 0.0
ledger.loc["IN_TRANSIT", "ending_coin"] = in_transit_coin

opening_value = (opening.opening_coin_position * opening.opening_fair_usdt).sum()
trade_cash = own.trade_cash_usdt.sum()
ending_value = ledger.ending_coin.sum() * final_fair
portfolio_pnl = trade_cash + ending_value - opening_value
completed_fee_coin = (complete.amount_sent - complete.amount_received).sum()

reconciliation = pd.Series({
    "opening_inventory_coin": opening.opening_coin_position.sum(),
    "net_trade_coin": own.signed_coin.sum(),
    "completed_transfer_fee_coin": completed_fee_coin,
    "ending_coin_including_transit": ledger.ending_coin.sum(),
    "opening_inventory_value_usdt": opening_value,
    "trade_cash_flow_after_fees_usdt": trade_cash,
    "ending_inventory_value_usdt": ending_value,
    "consolidated_portfolio_pnl_usdt": portfolio_pnl,
})
display(reconciliation.to_frame("value").round(6))

# Attribute portfolio PnL by location using the final global fair value as the
# internal transfer price. The source bears completed transfer quantity fees:
# it receives internal credit for amount_received, while amount_sent leaves.
# Unresolved sends are transferred at full amount to the IN_TRANSIT account.
locations = ledger.index
location_pnl = pd.DataFrame(index=locations)
location_pnl["trade_cash_usdt"] = (
    own.groupby("exchange")["trade_cash_usdt"].sum().reindex(locations).fillna(0)
)
location_pnl["internal_transfer_cash_usdt"] = 0.0
for row in complete.itertuples():
    transfer_value = row.amount_received * final_fair
    location_pnl.loc[row.exchange_from, "internal_transfer_cash_usdt"] += transfer_value
    location_pnl.loc[row.exchange_to, "internal_transfer_cash_usdt"] -= transfer_value
for row in sent_unresolved.itertuples():
    transfer_value = row.amount_sent * final_fair
    location_pnl.loc[row.exchange_from, "internal_transfer_cash_usdt"] += transfer_value
    location_pnl.loc["IN_TRANSIT", "internal_transfer_cash_usdt"] -= transfer_value

location_pnl["opening_value_usdt"] = 0.0
opening_values_by_exchange = (
    opening.assign(opening_value_usdt=opening.opening_coin_position * opening.opening_fair_usdt)
    .set_index("exchange")["opening_value_usdt"]
)
location_pnl.loc[opening_values_by_exchange.index, "opening_value_usdt"] = opening_values_by_exchange
location_pnl["ending_value_usdt"] = ledger["ending_coin"] * final_fair
location_pnl["portfolio_pnl_usdt"] = (
    location_pnl["trade_cash_usdt"]
    + location_pnl["internal_transfer_cash_usdt"]
    + location_pnl["ending_value_usdt"]
    - location_pnl["opening_value_usdt"]
)

print("Per-location portfolio PnL under final-fair-value internal transfer pricing:")
display(location_pnl.round(3))
print("Per-location PnL sum:", round(location_pnl.portfolio_pnl_usdt.sum(), 6))
print("Consolidated PnL:", round(portfolio_pnl, 6))
assert np.isclose(location_pnl.portfolio_pnl_usdt.sum(), portfolio_pnl)

print("Venue/location ending inventory:")
display(ledger[["opening_coin", "net_trade_coin", "completed_sent_coin",
                "completed_received_coin", "unresolved_sent_coin", "ending_coin"]].round(3))

,value
opening_inventory_coin,9.652352e+06
net_trade_coin,2.179274e+07
completed_transfer_fee_coin,1.845235e+05
ending_coin_including_transit,3.126057e+07
opening_inventory_value_usdt,2.726350e+04
trade_cash_flow_after_fees_usdt,-6.080676e+04
ending_inventory_value_usdt,1.108326e+05
consolidated_portfolio_pnl_usdt,2.276236e+04


Per-location portfolio PnL under final-fair-value internal transfer pricing:


,trade_cash_usdt,internal_transfer_cash_usdt,opening_value_usdt,ending_value_usdt,portfolio_pnl_usdt
location,,,,,
Exchange A,-287820.976,281681.961,1834.509,63196.351,55222.827
Exchange B,-45128.141,39189.003,-6549.818,-1821.934,-1211.254
Exchange C,128762.735,-95855.708,2018.187,-50852.021,-19963.180
Exchange D,99170.106,-62015.393,29420.871,-11152.304,-3418.462
Exchange E,44209.512,-45521.681,539.753,-6004.023,-7855.945
Exchange F,0.000,-90352.146,0.000,90340.517,-11.629
Exchange G,0.000,-23935.135,0.000,23935.135,0.000
Exchange H,0.000,0.000,0.000,0.000,0.000
IN_TRANSIT,0.000,-3190.900,0.000,3190.900,0.000


Per-location PnL sum: 22762.357414
Consolidated PnL: 22762.357414
Venue/location ending inventory:


,opening_coin,net_trade_coin,completed_sent_coin,completed_received_coin,unresolved_sent_coin,ending_coin
location,,,,,,
Exchange A,6.509035e+05,9.663551e+07,90292438.0,10830685.5,0.0,1.782466e+07
Exchange B,-2.318570e+06,1.288509e+07,65025888.0,54845488.9,900000.0,-5.138803e+05
Exchange C,7.144180e+05,-4.208354e+07,3094000.0,30120200.0,0.0,-1.434292e+07
Exchange D,1.041469e+07,-3.102809e+07,43546110.0,61013971.0,0.0,-3.145530e+06
Exchange E,1.909060e+05,-1.461624e+07,5490300.0,18222184.0,0.0,-1.693447e+06
Exchange F,0.000000e+00,0.000000e+00,29779610.0,55260339.0,0.0,2.548073e+07
Exchange G,0.000000e+00,0.000000e+00,0.0,6750954.1,0.0,6.750954e+06
Exchange H,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.000000e+00
IN_TRANSIT,0.000000e+00,0.000000e+00,0.0,0.0,0.0,9.000000e+05


### Results Interpretation

The transfer-aware consolidated reconciliation is:

| Component | Result |
|---|---:|
| Opening inventory | 9,652,351.5 COIN |
| Net COIN acquired through trades | +21,792,741.3 COIN |
| Completed transfer quantity fees | -184,523.5 COIN |
| Ending inventory including in-transit COIN | 31,260,569.3 COIN |
| Opening inventory value | 27,263.50 USDT |
| Trade cash flow after trading fees | -60,806.76 USDT |
| Ending inventory value | 110,832.62 USDT |
| **Consolidated portfolio PnL** | **+22,762.36 USDT** |

The consolidated result is lower than the earlier non-transfer-aware result because completed transfer fees destroy 184,523.5 COIN. The unresolved 900,000-COIN transfer is still an owned asset under the base case, so it is removed from Exchange B and recorded as inventory in transit rather than treated as a loss.

Venue balances are fully reconciled for Exchanges A–G and the in-transit account. Per-location portfolio PnL requires an internal transfer-pricing convention, so I use the **final global fair value** for every internal movement:

- for completed transfers, the source receives internal USDT credit for `amount_received × final fair value`, the destination receives an equal debit, and the source therefore bears the quantity fee;
- for the unresolved transfer, Exchange B receives credit for the full amount sent and `IN_TRANSIT` receives the equal debit;
- internal transfer credits and debits sum to zero, so location PnL must reconcile exactly to consolidated PnL.

| Location | Portfolio PnL (USDT) | Interpretation |
|---|---:|---|
| Exchange A | +55,222.83 | Largest positive location contribution |
| Exchange B | -1,211.25 | Includes source-side transfer-fee attribution and unresolved send |
| Exchange C | -19,963.18 | Largest negative trading-venue portfolio contribution |
| Exchange D | -3,418.46 | Negative full-period portfolio contribution |
| Exchange E | -7,855.95 | Negative full-period portfolio contribution |
| Exchange F | -11.63 | Transfer-fee effect on a non-trading location |
| Exchange G | 0.00 | Internal movements priced at the final mark |
| Exchange H | 0.00 | Contradictory receipt-only records excluded from the base ledger |
| `IN_TRANSIT` | 0.00 | Asset and internal transfer debit offset at the same final mark |
| **Consolidated** | **+22,762.36** | **Exactly equals the consolidated cash-and-inventory calculation** |

These per-location values are coherent under the stated convention, but their allocation is convention-dependent. The consolidated PnL is invariant to the internal transfer price as long as internal credits and debits cancel and transfer fees are recognized consistently.


### Question 2: COIN trading profitability analysis

In [7]:
execution = own.groupby("exchange")["execution_pnl_usdt"].agg(
    execution_pnl_usdt="sum", trades="size", pnl_per_trade_usdt="mean"
)
execution.loc["Total"] = [own.execution_pnl_usdt.sum(), len(own), own.execution_pnl_usdt.mean()]
display(execution.round(4))
print("Execution PnL by side:")
display(own.pivot_table(index="exchange", columns="own_side", values="execution_pnl_usdt", aggfunc="sum").round(3))

,execution_pnl_usdt,trades,pnl_per_trade_usdt
exchange,,,
Exchange A,1602.9489,3676.0,0.4361
Exchange B,3703.3184,8136.0,0.4552
Exchange C,1365.3138,3366.0,0.4056
Exchange D,3391.0602,13833.0,0.2451
Exchange E,1273.0383,4921.0,0.2587
Total,11335.6797,33932.0,0.3341


Execution PnL by side:


own_side,buy,sell
exchange,,
Exchange A,1586.581,16.368
Exchange B,2987.865,715.454
Exchange C,1422.157,-56.843
Exchange D,923.903,2467.157
Exchange E,994.892,278.146


### Results Interpretation

#### Overall conclusion

Yes. COIN trading was profitable overall and on every trading exchange when execution is measured against each venue's contemporaneous fair value after trading fees. The strategy generated **+11,335.68 USDT** across **33,932 own trades**, equivalent to approximately **+0.3341 USDT per trade**.

This is a strong cross-sectional result because profitability is not dependent on a single venue: all five trading exchanges contribute positively.

#### Execution PnL by exchange

| Rank by total PnL | Exchange | Execution PnL (USDT) | Own trades | PnL per trade (USDT) | Share of total PnL |
|---:|---|---:|---:|---:|---:|
| 1 | Exchange B | +3,703.32 | 8,136 | +0.4552 | 32.7% |
| 2 | Exchange D | +3,391.06 | 13,833 | +0.2451 | 29.9% |
| 3 | Exchange A | +1,602.95 | 3,676 | +0.4361 | 14.1% |
| 4 | Exchange C | +1,365.31 | 3,366 | +0.4056 | 12.0% |
| 5 | Exchange E | +1,273.04 | 4,921 | +0.2587 | 11.2% |
|  | **Total** | **+11,335.68** | **33,932** | **+0.3341** | **100.0%** |

Exchange B is the strongest venue on both total PnL and average PnL per trade. It contributes nearly one-third of consolidated execution PnL with fewer trades than Exchange D. This combination of scale and unit economics makes Exchange B the highest-quality venue in this sample.

Exchange D contributes almost as much total PnL as Exchange B, but it requires approximately 70% more trades and earns only +0.2451 USDT per trade. Its result is therefore primarily volume-driven. The strategy is profitable there, but execution efficiency is materially weaker.

Exchange A and Exchange C have relatively high PnL per trade despite lower volumes. Exchange E is profitable but ranks near the bottom on both total and per-trade performance.

#### Buy-side and sell-side contribution

| Exchange | Buy-side PnL (USDT) | Sell-side PnL (USDT) | Main contributor | Interpretation |
|---|---:|---:|---|---|
| Exchange A | +1,586.58 | +16.37 | Buy side | Almost all execution edge comes from buys |
| Exchange B | +2,987.86 | +715.45 | Buy side | Strong and diversified across both sides |
| Exchange C | +1,422.16 | -56.84 | Buy side | Profitable overall, but sells require review |
| Exchange D | +923.90 | +2,467.16 | Sell side | Sell execution is the primary strength |
| Exchange E | +994.89 | +278.15 | Buy side | Both sides profitable; buys dominate |
| **Total** | **+7,915.40** | **+3,420.28** | **Buy side** | **Approximately 69.8% of PnL comes from buys** |

The side results reveal that aggregate profitability is not perfectly balanced. Buys generate approximately 69.8% of execution PnL, driven especially by Exchanges A, B, and C. Exchange D is the exception: its sell side produces most of its edge. Exchange C is the only exchange with a negative side-level result, losing approximately 56.84 USDT on sells, although profitable buys more than compensate for it.

The Exchange C sell result is small relative to consolidated PnL and does not justify disabling sells immediately. It should instead be broken down by time, size, taker direction, spread, and market regime to determine whether the loss is systematic or sample noise.

#### Total PnL versus efficiency

Total PnL and PnL per trade answer different questions:

- **Total execution PnL** measures the amount of value generated at the observed scale.
- **PnL per trade** measures average execution efficiency, but it does not control for trade size.

Consequently, Exchange D is economically important because of its volume, while Exchanges B and A appear more efficient on a per-trade basis. A complete venue-allocation decision should also examine PnL per unit of COIN, PnL per unit of risk, capacity, fill probability, and adverse-selection markouts. Average PnL per trade alone can be misleading if typical trade sizes differ across exchanges.

#### Relationship to portfolio PnL

Execution PnL must be distinguished from the full **+22,762.36 USDT** portfolio result:

| Measure | Result | What it measures |
|---|---:|---|
| Net execution PnL | +11,335.68 USDT | Trade prices versus contemporaneous venue fair values, after trading fees |
| Inventory holding/mark PnL | +11,426.68 USDT | Subsequent valuation effect of opening and accumulated inventory |
| Consolidated portfolio PnL | +22,762.36 USDT | Combined execution and inventory result under the stated accounting convention |

The strategy therefore made money through both execution quality and inventory exposure. Positive portfolio PnL is not solely proof of trading skill, while positive execution PnL on every venue is stronger evidence that the strategy generally traded at favorable prices.

#### Practical recommendations

1. Preserve trading on all five venues because each is execution-profitable in this sample.
2. Prioritize Exchange B where capacity permits, because it combines the highest total and per-trade PnL.
3. Investigate Exchange D's lower unit economics before increasing volume further.
4. Review Exchange C sells by regime rather than applying an unconditional restriction.
5. Add size-normalized and risk-normalized metrics before making capital-allocation decisions.
6. Validate venue and side results across additional weeks to distinguish persistent edge from sample-specific performance.


### Question 3: Transfers success analysis 
### Question 4: Data quality analysis

In [8]:
complete["delay_seconds"] = (complete.time_confirmed_received_utc - complete.time_sent_utc).dt.total_seconds()
complete["fee_coin_reconciles"] = np.isclose(complete.amount_sent - complete.amount_received, complete.fee)
complete["explicit_fee_usdt_at_send"] = complete.fee * complete.fair_value_usdt_sent
complete["send_to_receive_value_change_usdt"] = (
    complete.amount_received * complete.fair_value_usdt_recv
    - complete.amount_sent * complete.fair_value_usdt_sent
)
transfers["request_to_send_seconds"] = (transfers.time_sent_utc - transfers.time_request_utc).dt.total_seconds()

transfer_summary = pd.Series({
    "complete": transfers.status.eq("complete").sum(),
    "received_without_send": transfers.status.eq("received_without_send").sum(),
    "sent_not_received": transfers.status.eq("sent_not_received").sum(),
    "completion_rate": transfers.status.eq("complete").mean(),
    "completed_fee_coin": complete.fee.sum(),
    "explicit_fee_usdt_at_send": complete.explicit_fee_usdt_at_send.sum(),
    "send_to_receive_value_change_usdt": complete.send_to_receive_value_change_usdt.sum(),
    "median_delay_seconds": complete.delay_seconds.median(),
    "mean_delay_seconds": complete.delay_seconds.mean(),
    "max_delay_seconds": complete.delay_seconds.max(),
    "completed_fee_mismatches": (~complete.fee_coin_reconciles).sum(),
    "sent_before_request": transfers.request_to_send_seconds.lt(0).sum(),
})
display(transfer_summary.to_frame("value").round(6))

exceptions = transfers.loc[
    ~transfers.status.eq("complete") | transfers.request_to_send_seconds.lt(0),
    ["transfer_id", "exchange_from", "exchange_to", "status", "amount_sent",
     "amount_received", "fee", "request_to_send_seconds"]
]
display(exceptions)

,value
complete,157.000000
received_without_send,3.000000
sent_not_received,1.000000
completion_rate,0.975155
completed_fee_coin,184523.500000
explicit_fee_usdt_at_send,522.598489
send_to_receive_value_change_usdt,-52.911781
median_delay_seconds,157.000000
mean_delay_seconds,5108.073800
max_delay_seconds,86220.000000


,transfer_id,exchange_from,exchange_to,status,amount_sent,amount_received,fee,request_to_send_seconds
80,81,Exchange H,Exchange F,received_without_send,0.55,0.55,NaN,NaN
81,82,Exchange F,Exchange B,complete,1663063.00,1662926.00,137.0,-0.090790
85,86,Exchange H,Exchange F,received_without_send,0.49,0.49,NaN,NaN
95,96,Exchange H,Exchange F,received_without_send,0.48,0.48,NaN,NaN
142,143,Exchange F,Exchange B,complete,1397398.00,1397260.00,138.0,-0.065024
148,149,Exchange F,Exchange B,complete,1101906.00,1101768.00,138.0,-0.093080
160,161,Exchange B,Exchange D,sent_not_received,900000.00,NaN,NaN,180.000000


### Results Interpretation — Question 3

#### Overall assessment

Transfers were **mostly successful operationally**, but not fully successful, and they were **financially costly**. Most requests completed and all completed quantities reconcile to their reported fees. However, four lifecycle exceptions, a large unresolved transfer, long delay tails, and explicit transfer fees prevent the transfer process from being described as risk-free or costless.

#### Operational status

| Status | Records | Share of requests | Interpretation |
|---|---:|---:|---|
| Complete send and receipt lifecycle | 157 | 97.52% | Operationally completed in-window |
| Received without send timestamp | 3 | 1.86% | Contradictory/incomplete event history |
| Sent but not received | 1 | 0.62% | Unresolved inventory in transit |
| **Total** | **161** | **100.00%** |  |

A 97.52% recorded completion rate is high, but the economic importance of an exception depends on its size rather than only its count. The single unresolved record contains **900,000 COIN**, worth approximately **3,190.90 USDT** at the final fair value. It is therefore more material than the three receipt-without-send records, whose combined amount is only 1.52 COIN.

#### Transfer timing

| Completed-transfer timing metric | Result | Interpretation |
|---|---:|---|
| Median delay | 157 seconds | A typical transfer completes in about 2.6 minutes |
| Mean delay | 5,108 seconds | About 85 minutes; inflated by slow-tail observations |
| Maximum delay | 86,220 seconds | Approximately 23 hours 57 minutes |
| Difference between mean and median | 4,951 seconds | Strongly right-skewed delay distribution |

The large gap between the median and mean shows that the typical transfer is relatively quick but a minority experience very long delays. This tail matters to a multi-venue trading strategy because inventory in transit cannot necessarily be used to quote, hedge, or settle on the destination venue. Even when the COIN is ultimately received, a long delay can cause missed trades, temporary shortfalls, forced rebalancing, or additional hedging costs.

The nearly 24-hour maximum should be investigated individually. It may represent a genuine operational delay, manual intervention, blockchain congestion, an exchange processing issue, or a timestamp defect.

#### Quantity and fee reconciliation

| Completed-transfer quantity metric | Result |
|---|---:|
| Total amount sent | 237,228,300 COIN |
| Total amount received | 237,043,776.5 COIN |
| Sent-minus-received quantity | 184,523.5 COIN |
| Recorded transfer fees | 184,523.5 COIN |
| Completed transfers with unexplained mismatch | 0 |

For every completed transfer, the reduction from amount sent to amount received equals the recorded fee. This is strong internal evidence that the completed-transfer quantity records are consistent: there is no unexplained loss beyond recorded fees in the complete subset.

The aggregate fee equals approximately **0.0778%** of completed COIN sent. Although this percentage is small, repeated transfers make the absolute cost material.

#### Financial interpretation

| Financial measure | Result | Correct interpretation |
|---|---:|---|
| Explicit fee valued at send-time fair value | -522.60 USDT | Incremental, directly attributable transfer cost |
| Send-to-receipt fair-value change | -52.91 USDT | Fee plus COIN price movement during transit |
| Unresolved transfer value at final mark | 3,190.90 USDT | Inventory exposure, not a realized loss in the base case |

The **-52.91 USDT** send-to-receipt value change should not be interpreted as the complete economic cost of transfers. It compares different quantities at different fair values and therefore combines two effects:

1. the explicit COIN fee; and
2. the market movement in COIN between send and receipt.

The portfolio would remain exposed to COIN price movement even if the inventory were not transferred. Therefore, the **522.60-USDT explicit fee** is the cleaner incremental cost measure. The send-to-receipt value change remains useful for describing what happened to the marked value while assets were in transit, but it is not a pure transfer-cost estimate.

#### Accounting treatment of the unresolved transfer

The base case records transfer 161 as `IN_TRANSIT`:

- Exchange B is reduced by 900,000 COIN when the transfer is sent.
- No receiving venue is credited without confirmation.
- A separate in-transit asset of 900,000 COIN is recognized.
- Consolidated inventory is preserved until evidence shows receipt or loss.

This treatment avoids prematurely recognizing a loss and maintains inventory conservation. If the transfer were assumed completely lost, consolidated portfolio PnL would fall by approximately 3,190.90 USDT, from **22,762.36 USDT** to **19,571.46 USDT**.

#### Conclusion and actions

Operationally, the transfer process works in most cases but has meaningful tail and exception risk. Financially, transfers impose a measurable fee cost and reduce ending inventory. Recommended actions are:

1. reconcile transfer 161 with destination, custody, and blockchain records;
2. introduce alerts based on both transfer age and USDT value at risk;
3. monitor delay percentiles by route rather than relying only on the mean;
4. include explicit transfer fees in route-selection and rebalancing decisions;
5. define a formal accounting state for requested, sent, in-transit, received, failed, and written-off transfers.

### Results Interpretation — Question 4

#### Data-quality summary

| Finding | Records affected | Materiality | Impact on analysis | Base treatment |
|---|---:|---|---|---|
| Receipt timestamp but no send timestamp | 3 | Low financially; high logical inconsistency | Cannot reconstruct lifecycle or source-side inventory movement | Excluded from base inventory ledger |
| Sent but no confirmed receipt | 1 | High | Destination balance is unknown; 900,000 COIN remains unresolved | Recorded as `IN_TRANSIT` |
| Send timestamp before request timestamp | 3 | Low direct PnL impact | Exact event ordering is unreliable | Retained, flagged as timestamp anomaly |
| Missing fee | 4 | Potentially material for incomplete records | Exact transfer cost cannot be calculated | Excluded from precise fee/PnL reconciliation |
| Missing received amount | 1 | High | Cannot determine fee or destination quantity | Scenario analysis required |
| Maximum delay of 86,220 seconds | 1 extreme observation | Medium/high operational risk | Distorts mean delay and may affect inventory availability | Retained and investigated separately |
| Completed amount difference not equal to fee | 0 | None observed | Complete-subset quantity reconciliation is reliable | No adjustment needed |
| Received amount greater than sent | 0 | None observed | No impossible positive quantity creation in complete rows | No adjustment needed |

#### 1. Receipt without a send event

Transfers 81, 86, and 96 show a confirmed receipt but no send timestamp. This is logically incomplete because an asset cannot be received without leaving some source or external wallet. Possible explanations include a missing source event, deposits initiated outside the captured system, manual balance adjustments, or incorrect timestamp ingestion.

Their combined amount is only **1.52 COIN**, worth approximately **0.005 USDT** at the final fair value. The financial impact is negligible, but recognizing the receipts without corresponding sources would violate inventory conservation. They are therefore excluded from the base ledger and reported separately.

#### 2. Sent but not received

Transfer 161 was sent from Exchange B to Exchange D but has no confirmed receipt, received amount, or fee. This is the most financially material data-quality issue.

| Transfer 161 field | Value |
|---|---|
| Source | Exchange B |
| Destination | Exchange D |
| Amount sent | 900,000 COIN |
| Final marked value | 3,190.90 USDT |
| Base accounting state | Inventory in transit |
| Conservative downside state | Assumed lost |

The consolidated base-case PnL remains valid only under the assumption that ownership of the in-transit asset is retained. Venue-level balances are necessarily uncertain: Exchange B no longer holds the COIN, while Exchange D cannot be credited without evidence of receipt.

#### 3. Negative request-to-send intervals

Three transfers have send timestamps roughly 0.07–0.09 seconds before their request timestamps. The tiny magnitude suggests clock synchronization, timestamp precision, asynchronous logging, or event-ingestion ordering rather than genuine pre-request execution.

These records can remain in aggregate PnL and quantity reconciliation because the discrepancy is a fraction of a second and their send/receipt lifecycle is otherwise complete. They should not, however, be used for precise request-to-send latency measurement without correction.

#### 4. Long-delay outlier

The maximum completed transfer delay is 86,220 seconds, almost 24 hours. Because the median is only 157 seconds, this observation belongs to a very different operational regime. It may be legitimate, but it should not be summarized as typical behavior.

The outlier has several possible analytical impacts:

- temporarily unavailable destination inventory;
- increased price exposure while assets are in transit;
- potential missed trading opportunities;
- distortion of average delay statistics;
- ambiguity over when the receiving venue should recognize inventory.

Delay reporting should therefore include median, upper percentiles, maximum, and value-weighted exposure rather than only an average.

#### 5. Fee and quantity reliability

The completed subset is internally strong: all 157 complete records satisfy `amount_sent - amount_received = fee`. The four missing fees occur in incomplete or contradictory lifecycle records. This means the **522.60-USDT explicit fee estimate is reliable for completed transfers**, but it is not necessarily the total cost of all 161 requests.

#### Impact on conclusions

| Conclusion | Confidence | Reason |
|---|---|---|
| Trading execution was profitable on every exchange | High | Derived from trade data and unaffected by transfer exceptions |
| Completed transfer fees reconcile | High | All 157 complete records reconcile exactly |
| Most transfers completed | High | Direct lifecycle evidence for 157 records |
| Exact ending balance by venue | Medium | Sensitive to transfer 161 and excluded receipt-only records |
| Base consolidated portfolio PnL | Medium/high | Coherent if unresolved inventory remains owned in transit |
| Transfer 161 was lost | Low/no evidence | No receipt confirmation is not proof of economic loss |
| Mean transfer delay represents normal performance | Low | Strong skew and an extreme maximum make the mean unrepresentative |

The data-quality issues do not overturn the conclusion that trading execution and the base-case consolidated portfolio were profitable. They do reduce confidence in exact venue-level inventory, total transfer cost across incomplete records, and the operational status of the unresolved transfer.

#### Recommended data controls

1. Enforce a transfer state machine with valid event ordering.
2. Require non-null sent amount, received amount, and fee before marking a transfer complete.
3. Reconcile internal transfer IDs to custody or blockchain transaction identifiers.
4. Synchronize clocks and preserve event-ingestion timestamps separately from event timestamps.
5. Create automatic exceptions for negative latencies, excessive delays, and unresolved high-value transfers.
6. Maintain an explicit in-transit inventory account and a formal write-off process.
7. Report both count-based and value-weighted operational success rates.


### Question 5: The main drivers of PnL and the assumptions

In [9]:
total_trade_fees = own.fee_usdt.sum()
gross_execution_edge = own.execution_pnl_usdt.sum() + total_trade_fees

# Holding PnL is the residual under the stated accounting convention.
holding_pnl = portfolio_pnl - own.execution_pnl_usdt.sum()
drivers = pd.Series({
    "gross_execution_edge_before_trade_fees": gross_execution_edge,
    "trade_fees": -total_trade_fees,
    "net_execution_pnl": own.execution_pnl_usdt.sum(),
    "non_execution_residual_inventory_mark_and_transfer_effects": holding_pnl,
    "consolidated_portfolio_pnl": portfolio_pnl,
    "explicit_completed_transfer_fee_at_send_fv": -complete.explicit_fee_usdt_at_send.sum(),
})
display(drivers.to_frame("USDT").round(3))

unresolved_value = in_transit_coin * final_fair
received_without_send_coin = transfers.loc[transfers.status.eq("received_without_send"), "amount_received"].sum()
sensitivity = pd.Series({
    "base_case_pnl_inventory_in_transit_recognized": portfolio_pnl,
    "pnl_if_unresolved_transfer_is_assumed_lost": portfolio_pnl - unresolved_value,
    "unresolved_transfer_final_value": unresolved_value,
    "excluded_received_without_send_coin": received_without_send_coin,
    "effect_if_excluded_receipts_were_recognized": received_without_send_coin * final_fair,
})
display(sensitivity.to_frame("USDT_or_coin_as_labelled").round(3))

,USDT
gross_execution_edge_before_trade_fees,14898.494
trade_fees,-3562.814
net_execution_pnl,11335.680
non_execution_residual_inventory_mark_and_transfer_effects,11426.678
consolidated_portfolio_pnl,22762.357
explicit_completed_transfer_fee_at_send_fv,-522.598


,USDT_or_coin_as_labelled
base_case_pnl_inventory_in_transit_recognized,22762.357
pnl_if_unresolved_transfer_is_assumed_lost,19571.457
unresolved_transfer_final_value,3190.900
excluded_received_without_send_coin,1.520
effect_if_excluded_receipts_were_recognized,0.005


### Results Interpretation

The economic PnL decomposition is:

| Driver | USDT |
|---|---:|
| Gross execution edge before trading fees | +14,898.49 |
| Trading fees | -3,562.81 |
| Net execution PnL | +11,335.68 |
| Non-execution residual: inventory holding/mark PnL and transfer effects | +11,426.68 |
| **Consolidated portfolio PnL** | **+22,762.36** |

Net execution PnL and the non-execution residual contributed similar amounts to the final result. The residual contains inventory holding/mark PnL as well as transfer effects, so it should not be interpreted as a pure directional holding return. Trading fees consumed approximately 23.9% of gross execution edge, so fee control is economically significant. Completed transfer fees represent an additional explicit cost of approximately 522.60 USDT; this is already reflected in reduced ending inventory and must not be subtracted from portfolio PnL a second time.

#### Robust conclusions

- Execution PnL is positive overall and on every trading exchange.
- Completed transfers reduced inventory by exactly their reported fees.
- The portfolio is profitable under both the base case and the conservative unresolved-transfer-loss scenario.
- ETH-style execution analysis and portfolio accounting measure different sources of performance and should be reported separately.

#### Assumption-sensitive conclusions

- The exact inventory holding PnL depends on the global fair-value construction and final mark.
- If the unresolved 900,000 COIN were assumed completely lost, portfolio PnL would fall from approximately 22,762.36 to 19,571.46 USDT.
- Per-venue portfolio PnL depends on an internal transfer-pricing convention; venue execution PnL is the more robust comparison.
- Recognizing the three receipt-without-send rows would have a negligible consolidated value effect, but doing so would violate inventory conservation without evidence of their source.

Overall, COIN trading demonstrated positive execution edge, and the portfolio also benefited from inventory holding exposure. The strategy should retain its profitable execution logic while improving transfer-cost control, monitoring long transfer tails, and formalizing the accounting treatment of inventory in transit.